In [2]:
# %% [markdown]
# # Portfolio Analyzer: yfinance API Integration & Industry Tracking
# This script reads a Quicken holdings export, pings yfinance for live
# asset classification and sector data, and calculates precise portfolio exposure.

# %%
import io
import re
import time
import pandas as pd
import numpy as np
import yfinance as yf

CSV_FILENAME = "data/Chris's Finances - Investing - Portfolio Value - By Managed Account - 2026-09-24.csv"

# %%
def load_and_parse_holdings(filepath):
    """Parses the Quicken 'Portfolio Value' export into a clean Account/Symbol DataFrame."""
    try:
        with open(filepath, 'rb') as f:
            raw_bytes = f.read()
    except FileNotFoundError:
        print(f"Error: Could not find '{filepath}'.")
        return None

    text = raw_bytes.decode('utf-8-sig', errors='replace').replace('\ufeff', '')
    lines = text.splitlines(keepends=True)

    header_idx = next((i for i, line in enumerate(lines) if "symbol" in line.lower() and "market value" in line.lower()), None)
    if header_idx is None:
        print("Error: Could not find table header row.")
        return None

    df = pd.read_csv(io.StringIO("".join(lines[header_idx:])), sep=',', on_bad_lines='skip')

    col_names = list(df.columns)
    col_names[0] = 'Name'
    df.columns = col_names

    if 'Market Value' in df.columns:
        df['Market Value'] = df['Market Value'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
        df['Market Value'] = pd.to_numeric(df['Market Value'], errors='coerce')

    parsed_data = []
    current_account = "Unknown Account"

    for _, row in df.iterrows():
        raw_name = str(row['Name'])
        if pd.isna(row['Name']) or raw_name.strip() == '':
            continue

        # Top-level accounts in Quicken have NO leading spaces. Holdings are indented.
        if not raw_name.startswith(' ') and not raw_name.startswith('\t'):
            current_account = raw_name.strip()
        else:
            # It's an indented holding, even if the symbol is completely blank
            name_val = raw_name.strip()

            # Handle missing symbols for Munis/Cash without creating a new account
            if pd.isna(row['Symbol']):
                symbol = "CASH" if "cash" in name_val.lower() else "UNLISTED_ASSET"
            else:
                symbol = str(row['Symbol']).strip().upper()

            market_value = row['Market Value'] if not pd.isna(row['Market Value']) else 0.0

            parsed_data.append({
                'Account': current_account,
                'Asset_Name': name_val,
                'Symbol': symbol,
                'Market_Value': market_value
            })

    return pd.DataFrame(parsed_data)

# %%

def fetch_yfinance_metadata(df):
    """
    Pings yfinance for live metadata. Relies on Quicken names for accuracy,
    detects Min Volatility / Muni strategies, and skips unlisted assets.
    """
    print("Pinging yfinance for live asset metadata (with rate-limit protection)...")

    # 1. Remap outdated tickers and catch unlisted assets BEFORE pinging
    TICKER_REMAP = {
        'ERJ': 'EMBJ',
        'ABC': 'COR',
        '*BDANOW': 'CASH',
        'USD=': 'CASH',
        'VSCO': 'VSCO',
        'IAC': 'IAC',
        'BK': 'BK',
        'UNLISTED_ASSET': 'UNLISTED_ASSET'
    }
    df['Symbol'] = df['Symbol'].replace(TICKER_REMAP)

    unique_symbols = [sym for sym in df['Symbol'].unique() if sym != "CASH" and str(sym) != "NAN"]

    metadata_dict = {}

    for sym in unique_symbols:
        # 2. Automatically skip unlisted assets, CUSIPs (9 chars), and internal IDs (M12345)
        if sym == 'UNLISTED_ASSET' or (len(sym) == 9 and sym.isalnum()) or bool(re.match(r'^M\d{5,}', sym)) or sym.endswith('-'):
            metadata_dict[sym] = {'Asset_Class': 'Municipal Bond', 'Sector_Category': 'Individual Bond / Unlisted'}
            continue

        max_retries = 3
        for attempt in range(max_retries):
            try:
                # Force known glitching tickers to individual equity without pinging if needed
                if sym in ['VSCO', 'IAC', 'BK']:
                    metadata_dict[sym] = {'Asset_Class': 'Individual Security', 'Sector_Category': 'Consumer / Financial (Manual)'}
                    break

                info = yf.Ticker(sym).info

                if not info or 'quoteType' not in info:
                    raise ValueError(f"Empty info returned for {sym}")

                quote_type = info.get('quoteType', 'UNKNOWN')

                if quote_type == 'EQUITY':
                    asset_class = 'Individual Security'
                    category = f"{info.get('sector', 'Unknown Sector')} - {info.get('industry', 'Unknown Industry')}"
                elif quote_type in ['ETF', 'MUTUALFUND']:
                    asset_class = 'ETF / Index'
                    category = info.get('category', 'Unknown Fund Category')
                else:
                    asset_class = 'Other'
                    category = quote_type

                metadata_dict[sym] = {'Asset_Class': asset_class, 'Sector_Category': category}
                break

            except Exception:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    metadata_dict[sym] = {'Asset_Class': 'Unknown', 'Sector_Category': 'API Error / Not Found'}

        time.sleep(0.3)

    # 3. Map fetched data back to the dataframe
    df['Asset_Class'] = df['Symbol'].apply(lambda x: 'Cash' if x == 'CASH' else metadata_dict.get(x, {}).get('Asset_Class', 'Unknown'))
    df['Sector_Category'] = df['Symbol'].apply(lambda x: 'Cash Equivalent' if x == 'CASH' else metadata_dict.get(x, {}).get('Sector_Category', 'Unknown'))

    # 4. Keyword Override: Force correct mapping based on the actual Quicken Name
    def override_category(row):
        name = str(row['Asset_Name']).lower()
        ac = str(row['Asset_Class'])
        cat = str(row['Sector_Category'])

        # Override for Min Vol / Low Volatility strategies
        if 'min vol' in name or 'low volatility' in name:
            ac = 'ETF / Index'
            cat = 'Low Volatility Equity'
        # Override for Municipal Bonds
        elif 'muni' in name:
            ac = 'Municipal Bond'
            cat = 'Municipal Fixed Income'

        return pd.Series([ac, cat])

    df[['Asset_Class', 'Sector_Category']] = df.apply(override_category, axis=1)

    return df

# %%
def analyze_portfolio_with_sectors(df):
    """Outputs the portfolio breakdown including Sector/Industry analysis."""
    if df is None or df.empty:
        return

    accounts = df['Account'].unique()

    print("\n" + "="*70)
    print("PORTFOLIO HOLDINGS & SECTOR EXPOSURE ANALYSIS")
    print("="*70)

    for acc in accounts:
        acc_df = df[df['Account'] == acc].copy()
        total_value = acc_df['Market_Value'].sum()

        if total_value == 0:
            continue

        acc_df['Weight_%'] = (acc_df['Market_Value'] / total_value) * 100

        # Determine True Underlying Asset Allocation (Macro Split)
        def classify_macro(row):
            cat = str(row['Sector_Category']).lower()
            name = str(row['Asset_Name']).lower()
            ac = str(row['Asset_Class']).lower()

            # If it says bond, muni, cash, or fixed income anywhere, it's safe money
            if 'muni' in cat or 'bond' in cat or 'fixed income' in cat or 'cash' in ac or 'muni' in name:
                return 'Fixed Income / Cash'
            # Otherwise, whether it's an individual stock or an equity ETF, it's equity
            return 'Equity'

        acc_df['Macro_Class'] = acc_df.apply(classify_macro, axis=1)

        # Calculate True Portfolio Ratios
        macro_mix = acc_df.groupby('Macro_Class')['Weight_%'].sum().to_dict()
        true_equity_weight = macro_mix.get('Equity', 0)
        true_fixed_weight = macro_mix.get('Fixed Income / Cash', 0)

        # Determine smarter benchmarks based on TRUE Equity vs Fixed Income ratio
        if true_equity_weight > 90:
            benchmark = "SPY (S&P 500) or ACWI (Global Equity) - 100% Equity"
        elif true_equity_weight > 70:
            benchmark = "AOA (80/20 Aggressive Growth Allocation)"
        elif true_equity_weight > 50:
            benchmark = "AOR (60/40 Core Growth Allocation)"
        elif true_fixed_weight > 80:
            benchmark = "MUB (Muni Bonds) or AGG (Core US Bond) - Fixed Income"
        else:
            benchmark = "AOM (40/60 Conservative Allocation)"

        print(f"\nACCOUNT: {acc}")
        print(f"Total Value:        ${total_value:,.2f}")
        # Print the true risk profile instead of just structural wrappers
        print(f"Macro Allocation:   {true_equity_weight:.1f}% Total Equity | {true_fixed_weight:.1f}% Fixed Income & Cash")
        print(f"Target Benchmark:   {benchmark}")
        print("-" * 50)

        # Display Top Holdings
        top_holdings = acc_df.sort_values(by='Market_Value', ascending=False).head(3)
        print("Top Holdings:")
        for _, row in top_holdings.iterrows():
            asset_name = str(row['Asset_Name'])[:25]
            sector_cat = str(row['Sector_Category']).replace('Unknown Fund Category', 'Fund (Inferred via Name)')[:35]
            print(f"  - {row['Symbol']:<6} | {asset_name:<25} | {row['Weight_%']:>4.1f}% | {sector_cat}")

        # Display the aggregate Sector / Strategy exposure
        print("\nPrimary Sector / Fund Strategy Exposure:")
        sector_exposure = acc_df.groupby('Sector_Category')['Weight_%'].sum().sort_values(ascending=False).head(5)
        for sector, weight in sector_exposure.items():
            if weight > 1.0:
                print(f"  > {sector[:40]:<40} : {weight:>4.1f}%")


# %%
if __name__ == "__main__":
    raw_df = load_and_parse_holdings(CSV_FILENAME)
    if raw_df is not None:
        enriched_df = fetch_yfinance_metadata(raw_df)
        analyze_portfolio_with_sectors(enriched_df)

Pinging yfinance for live asset metadata (with rate-limit protection)...

PORTFOLIO HOLDINGS & SECTOR EXPOSURE ANALYSIS

ACCOUNT: Fidelity TRUST - Alex - Managed
Total Value:        $1,568,374.40
Macro Allocation:   82.7% Total Equity | 17.3% Fixed Income & Cash
Target Benchmark:   AOA (80/20 Aggressive Growth Allocation)
--------------------------------------------------
Top Holdings:
  - USMV   | ISHARES EDGE MSCI MIN VOL | 10.8% | Low Volatility Equity
  - FSMNX  | FIDELITY SAI MUNICIPAL IN |  7.8% | Municipal Fixed Income
  - EEMV   | ISHARES EDGE MSCI MIN VOL |  5.0% | Low Volatility Equity

Primary Sector / Fund Strategy Exposure:
  > Low Volatility Equity                    : 20.5%
  > Unknown Fund Category                    : 13.0%
  > Municipal Fixed Income                   :  8.3%
  > Technology - Semiconductors              :  7.4%
  > Short-Term Inflation-Protected Bond      :  4.3%

ACCOUNT: Fidelity TRUST - Chris - Managed
Total Value:        $1,601,590.04
Macro Allocat

In [3]:
# %%
from IPython.display import display, HTML

def generate_interactive_portfolio_view(df):
    """
    Groups the portfolio by Account, then by Asset Class and Sector,
    creating a nested, visually rich table in Jupyter to analyze individual holdings.
    """
    if df is None or df.empty:
        print("No data available to display.")
        return

    # Create a copy and sort the data for a logical visual hierarchy
    view_df = df.copy()
    view_df = view_df.sort_values(by=['Account', 'Asset_Class', 'Sector_Category', 'Market_Value'],
                                  ascending=[True, True, True, False])

    # Calculate account weights for the display
    view_df['Account_Total'] = view_df.groupby('Account')['Market_Value'].transform('sum')
    view_df['Weight_%'] = (view_df['Market_Value'] / view_df['Account_Total']) * 100

    # Format the numbers as clean strings for the final table
    view_df['Weight_%'] = view_df['Weight_%'].map('{:.1f}%'.format)
    view_df['Market_Value'] = view_df['Market_Value'].apply(lambda x: f"${x:,.2f}")

    # Group the dataframe into a MultiIndex hierarchical structure
    hierarchical_df = view_df.set_index(['Account', 'Asset_Class', 'Sector_Category', 'Symbol'])

    # Keep only the relevant columns for the final display
    display_df = hierarchical_df[['Asset_Name', 'Market_Value', 'Weight_%']]

    # Apply pandas styling to make the table look polished in Jupyter
    styled_table = (display_df.style
                    .set_caption("<b style='font-size:16px;'>Detailed Portfolio Exposure (Grouped by Sector/Strategy)</b>")
                    .set_table_styles([{
                        'selector': 'th.row_heading',
                        'props': [('text-align', 'left'), ('vertical-align', 'top'), ('border-bottom', '1px solid #ddd')]
                    }, {
                        'selector': 'td',
                        'props': [('text-align', 'left'), ('border-bottom', '1px solid #f0f0f0')]
                    }])
                   )

    # Render the interactive HTML table directly in the notebook output
    display(styled_table)

# Execute the new view
generate_interactive_portfolio_view(enriched_df)

In [4]:
# Force display to show in the cell output
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

# Check if enriched_df has rows
print(f"Total rows in enriched_df: {len(enriched_df)}")

# Generate and print directly
hierarchical_df = enriched_df.set_index(['Account', 'Asset_Class', 'Sector_Category', 'Symbol'])
print(hierarchical_df[['Asset_Name', 'Market_Value']].to_string())

Total rows in enriched_df: 753
                                                                                                                                                                                                                                         Asset_Name  Market_Value
Account                          Asset_Class         Sector_Category                                         Symbol                                                                                                                                              
Fidelity TRUST - Alex - Managed  Individual Security Technology - Semiconductors                             AMD                                                                                                         ADVANCED MICRO DEVICES INC       5789.86
                                                     Financial Services - Asset Management                   AMG                                                                                   

In [6]:
# %%
excel_output_path = "portfolio_holdings_analysis.xlsx"

with pd.ExcelWriter(excel_output_path, engine='openpyxl') as writer:
    # 1. Master sheet with all 753 enriched rows
    enriched_df.to_excel(writer, sheet_name="All Holdings", index=False)

    # 2. Account-specific breakdown tabs
    for acc in enriched_df['Account'].unique():
        acc_df = enriched_df[enriched_df['Account'] == acc].copy()

        # Calculate weights within account
        total_val = acc_df['Market_Value'].sum()
        if total_val > 0:
            acc_df['Weight_%'] = (acc_df['Market_Value'] / total_val) * 100
        else:
            acc_df['Weight_%'] = 0.0

        # Sort hierarchically: Asset Class -> Sector/Category -> Market Value
        acc_df = acc_df.sort_values(
            by=['Asset_Class', 'Sector_Category', 'Market_Value'],
            ascending=[True, True, False]
        )

        # Clean Excel tab name (Excel limits sheet names to 31 chars and bans certain symbols)
        clean_sheet_name = str(acc)[:28].replace(':', '').replace('/', '').replace('\\', '').replace('?', '').replace('*', '')

        cols_to_save = ['Symbol', 'Asset_Name', 'Asset_Class', 'Sector_Category', 'Market_Value', 'Weight_%']
        acc_df[cols_to_save].to_excel(writer, sheet_name=clean_sheet_name, index=False)

print(f"Export complete. File saved to: {excel_output_path}")

Export complete. File saved to: portfolio_holdings_analysis.xlsx
